In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from torchvision.transforms.functional import to_tensor

sys.path.append("..")
from src import *

# Compute Metrics

In [2]:
dataset = ObjaverseDataset3D()

In [3]:
TESTSET_DIR = Path("dataset/test")
REND_DIR = Path("renderings")
TEST_DIR = Path("tests")
dirs = [
    "gt",
    "sd15_mlsd",
    "sd15_ours",
    "sdxl_ours",
    "sdxl_mlsd_llite",
    "sdxl_ours_llite",
]
testset = pd.read_json(TESTSET_DIR / "metadata.jsonl", orient="records", lines=True)
testset.index = pd.Series(testset.uv_file_name.map(lambda x: Path(x).stem), name="uid")

In [4]:
metrics: dict[str, Metric] = {
    "psnr_t": PSNRMetric(need_renders=False),
    "psnr_r": PSNRMetric(need_renders=True),
    "ssim_t": SSIMMetric(need_renders=False),
    "ssim_r": SSIMMetric(need_renders=True),
    "lpips_t": LPIPSMetric(need_renders=False),
    "lpips_r": LPIPSMetric(need_renders=True),
    "fid_t": FIDMetric(need_renders=False),
    "fid_r": FIDMetric(need_renders=True),
    "clipiqa_t": CLIPIQAMetric(need_renders=False),
    "clipiqa_r": CLIPIQAMetric(need_renders=True),
    "clip_t": CLIPMetric(need_renders=False),
    "clip_r": CLIPMetric(need_renders=True),
    "brisque_t": BRISQUEMetric(need_renders=False),
    "brisque_r": BRISQUEMetric(need_renders=True),
}

In [5]:
def path2tensor(path: Path) -> torch.Tensor:
    with Image.open(path) as img:
        img = img.resize((512, 512))
        if img.mode in ("RGBA", "LA"):
            white_bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
            img = Image.alpha_composite(white_bg, img.convert("RGBA"))
        return to_tensor(img.convert("RGB")).unsqueeze(0)

In [6]:
def compute_metrics(tag, testset):
    views = 3
    uids = testset.index
    y_tex = torch.empty((len(uids), 3, 512, 512))
    gt_tex = torch.empty_like(y_tex)
    y_ren = torch.empty((len(uids), views, 3, 512, 512))
    gt_ren = torch.empty_like(y_ren)
    captions = []

    cprint("yellow:Preprocessing testset...")
    for i, uid in tqdm(enumerate(uids)):
        y_tex[i] = path2tensor(TEST_DIR / tag / f"{uid}.png")
        gt_tex[i] = path2tensor(TESTSET_DIR / "diffuse" / f"{uid}.png")
        for view in range(views):
            y_ren[i, view] = path2tensor(REND_DIR / tag / f"{uid[:-2]}_{view}.png")
            gt_ren[i, view] = path2tensor(REND_DIR / "gt" / f"{uid[:-2]}_{view}.png")
        captions.append(testset.loc[uid].caption)

    cprint(f"yellow:Computing metrics for ({tag})...")
    for k, metric in metrics.items():
        if metric.need_renders:
            m = metric(y_ren, gt_ren, captions)
        else:
            m = metric(y_tex, gt_tex, captions)
        cprint(f"green:{k}", f"blue:{m:.4f}")

In [7]:
# compute_metrics("sd15_mlsd", testset=testset)
# compute_metrics("sd15_ours", testset=testset)
# compute_metrics("sdxl_ours", testset=testset)
compute_metrics("sdxl_mlsd_llite", testset=testset)
compute_metrics("sdxl_ours_llite", testset=testset)

Preprocessing testset...


0it [00:00, ?it/s]

98it [00:04, 20.02it/s]


Computing metrics for (sdxl_mlsd_llite)...
psnr_t 8.7784
psnr_r 20.4701
ssim_t 0.2979
ssim_r 0.8810
lpips_t 0.7830
lpips_r 0.1484
fid_t 244.6052
fid_r 112.0747
clipiqa_t 0.7776


clipiqa_r 0.8288
clip_t 0.2673


clip_r 0.2430
brisque_t 33.1969
brisque_r 75.6597
Preprocessing testset...


98it [00:04, 21.66it/s]


Computing metrics for (sdxl_ours_llite)...
psnr_t 8.9395
psnr_r 20.5552
ssim_t 0.3499
ssim_r 0.8946
lpips_t 0.7847
lpips_r 0.1309
fid_t 264.4204
fid_r 93.2355
clipiqa_t 0.7981


clipiqa_r 0.8293
clip_t 0.2643


clip_r 0.2410
brisque_t 41.1915
brisque_r 76.4239


### Texture-based metrics

| Model             |  $\text{PSNR} ↑$ |  $\text{SSIM} ↑$ | $\text{LPIPS} ↓$ |   $\text{FID} ↓$   | $\text{CLIP-IQA} ↑$ |  $\text{CLIP} ↑$ | $\text{BRISQUE} ↓$ |
| ----------------- | :--------------: | :--------------: | :--------------: | :----------------: | :-----------------: | :--------------: | :----------------: |
| `sd15_mlsd`       |      $7.971$     |      $0.244$     |      $0.812$     |      $233.044$     |       $0.809$       |      $0.251$     |  $\mathbf{37.678}$ |
| `sd15_ours`       |      $8.438$     |      $0.284$     |      $0.789$     | $\mathbf{229.223}$ |       $0.812$       |      $0.244$     |      $43.121$      |
| `sdxl_mlsd_llite` |      $8.778$     |      $0.298$     | $\mathbf{0.783}$ | $\mathbf{244.605}$ |       $0.778$       | $\mathbf{0.267}$ |  $\mathbf{33.197}$ |
| `sdxl_ours_llite` | $\mathbf{8.940}$ | $\mathbf{0.350}$ |      $0.785$     |      $264.420$     |   $\mathbf{0.798}$  |      $0.264$     |      $41.192$      |

### Rendering-based metrics

| Model             |  $\text{PSNR} ↑$  |  $\text{SSIM} ↑$ | $\text{LPIPS} ↓$ |   $\text{FID} ↓$  | $\text{CLIP-IQA} ↑$ |  $\text{CLIP} ↑$ | $\text{BRISQUE} ↓$ |
| ----------------- | :---------------: | :--------------: | :--------------: | :---------------: | :-----------------: | :--------------: | :----------------: |
| `sd15_mlsd`       |      $19.041$     |      $0.867$     |      $0.158$     |     $113.609$     |       $0.826$       |      $0.236$     |  $\mathbf{74.906}$ |
| `sd15_ours`       |      $19.912$     |      $0.879$     |      $0.147$     |     $100.043$     |       $0.826$       |      $0.239$     |      $78.941$      |
| `sdxl_mlsd_llite` |      $20.470$     |      $0.881$     |      $0.148$     |     $112.075$     |       $0.829$       | $\mathbf{0.243}$ |  $\mathbf{75.660}$ |
| `sdxl_ours_llite` | $\mathbf{20.555}$ | $\mathbf{0.895}$ | $\mathbf{0.131}$ | $\mathbf{93.236}$ |   $\mathbf{0.829}$  |      $0.241$     |      $76.424$      |
